# Masonry blocks from a dual mesh

In [ ]:
%pip install -q compas==2.15.1 pythreejs==2.4.2
%pip install -q compas_notebook==0.11.0 --no-deps

In [ ]:
from compas.datastructures import Mesh
from compas.geometry import Line, Plane, bestfit_plane_numpy
from compas.itertools import pairwise
from compas_notebook import Viewer
from compas_notebook.config import Config

## Dual of a vault

In [ ]:
mesh = Mesh.from_meshgrid(dx=10, nx=8)
for vertex in mesh.vertices():
    x, y, _ = mesh.vertex_coordinates(vertex)
    mesh.vertex_attribute(vertex, "z", 3 * x * (10 - x) * y * (10 - y) / 625)
mesh.quads_to_triangles()
dual = mesh.dual(include_boundary=True)
dual.flip_cycles()

## One block per face

In [ ]:
thickness = 0.3
blocks = []
for face in dual.faces():
    middle = dual.face_polygon(face)
    normals = [dual.vertex_normal(vertex) for vertex in dual.face_vertices(face)]
    bottom = [point - normal * (0.5 * thickness) for point, normal in zip(middle, normals)]
    top = [point + normal * (0.5 * thickness) for point, normal in zip(middle, normals)]
    plane = Plane(*bestfit_plane_numpy(top))
    top = [plane.intersection_with_line(Line(a, b)) for a, b in zip(bottom, top)]
    sides = [[a, b, bb, aa] for (a, b), (aa, bb) in zip(pairwise(bottom + bottom[:1]), pairwise(top + top[:1]))]
    blocks.append(Mesh.from_polygons([bottom[::-1], top] + sides))
len(blocks)

## Scene

In [ ]:
config = Config()
config.view.width, config.view.height = 700, 360
config.view.camera.position = [5, -9, 9]
config.view.camera.target = [5, 5, 1]
viewer = Viewer(config=config)
for block in blocks:
    viewer.scene.add(block, color="#c78558", show_edges=True)
viewer.show()